In [ ]:
# Initialization, styling configuration, and helper utilities
%matplotlib inline

import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from ipywidgets import fixed, interact

# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/ML-DL/02_deep_learning/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp

import importlib
importlib.reload(hp)

hp.setup_theme()

# Sequence Models & Attention
## Recurrent Architectures, The Information Bottleneck, and Bahdanau Attention

This lecture traces the historical evolution of sequence modeling:
- **Sequential Data**: Temporal dependencies and autoregressive state modeling
- **Recurrent Neural Networks (RNN)**: The Elman architecture and state recurrence
- **Backpropagation Through Time (BPTT)**: Vanishing and exploding gradients
- **Gated Memory**: Long Short-Term Memory (LSTM) and Gated Recurrent Units (GRU)
- **Sequence-to-Sequence (Seq2Seq)**: Encoder-decoder structures and the vector bottleneck
- **The Attention Breakthrough**: Bahdanau Additive Attention (2014) as the historical bridge to Transformers

# Recurrent Neural Networks (RNN)
## The Elman Architecture

Sequential data (text, audio, time series) consists of ordered observations $\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_T$ of variable length.

Standard feedforward networks cannot maintain temporal state.
An **Elman Recurrent Neural Network (RNN)** maintains an internal hidden state vector $\mathbf{h}_t$ that acts as a recurrent memory of past inputs:

$$\mathbf{h}_t = \tanh\left(\mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{W}_{xh} \mathbf{x}_t + \mathbf{b}_h\right)$$
$$\mathbf{y}_t = g\left(\mathbf{W}_{hy} \mathbf{h}_t + \mathbf{b}_y\right)$$

where:
- $\mathbf{W}_{hh} \in \mathbb{R}^{H \times H}$ is the recurrent state transition matrix (shared across all time steps)
- $\mathbf{W}_{xh} \in \mathbb{R}^{H \times D}$ is the input projection matrix
- $\mathbf{h}_{t-1}$ contains the summary of history up to step $t-1$

## Backpropagation Through Time (BPTT)
## The Vanishing and Exploding Gradient Dilemma

To train an RNN, we unroll the computation graph across $T$ time steps.
Differentiating loss $E$ with respect to recurrent weights $\mathbf{W}_{hh}$ involves computing:
$$\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_1} = \prod_{t=2}^T \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \prod_{t=2}^T \mathbf{W}_{hh}^\mathrm{T} \mathrm{diag}\left(1 - \mathbf{h}_t^2\right)$$

Let $\rho$ be the spectral radius (largest absolute eigenvalue) of $\mathbf{W}_{hh}$:
- **Exploding Gradients ($\rho > 1$)**: Gradients grow exponentially $\sim \rho^T$, causing numerical overflows (mitigated via gradient clipping).
- **Vanishing Gradients ($\rho < 1$)**: Gradients decay exponentially to zero, preventing standard RNNs from learning long-term dependencies beyond $\sim 10$ steps.

# Long Short-Term Memory (LSTM)
## Constant Error Carousels & Gated Memory

To overcome vanishing gradients, Sepp Hochreiter and Jürgen Schmidhuber (1997) introduced the **Long Short-Term Memory (LSTM)** architecture.

LSTM decouples the hidden state $\mathbf{h}_t$ from an explicit **memory cell state** $\mathbf{c}_t$, controlled by three multiplicative gates:
1. **Forget Gate**: Decides what information to discard from the cell:
   $$\mathbf{f}_t = \sigma\left(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f\right)$$
2. **Input Gate & Candidate Cell**: Computes new candidate information:
   $$\mathbf{i}_t = \sigma\left(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i\right), \quad \tilde{\mathbf{c}}_t = \tanh\left(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c\right)$$
3. **Additive Cell Update**: Linear highway preserving gradient flow:
   $$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$
4. **Output Gate**:
   $$\mathbf{o}_t = \sigma\left(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o\right), \quad \mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

<small>📖 *Source: [Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. Neural Computation, 9(8), 1735–1780](https://doi.org/10.1162/neco.1997.9.8.1735)*</small>

# Sequence-to-Sequence (Seq2Seq)
## The Fixed-Vector Information Bottleneck

In Sequence-to-Sequence tasks (e.g. machine translation), an **Encoder RNN** compresses an arbitrary-length source sentence $\mathbf{x}_1, \dots, \mathbf{x}_T$ into a single fixed-length vector $\mathbf{h}_T$:

```
Encoder: x_1 -> x_2 -> x_3 -> ... -> x_T  ===> [ Fixed Vector h_T ] ===> Decoder: y_1 -> y_2 -> ...
```

### The Fundamental Bottleneck
- For long sentences ($T > 20$ words), compressing all syntactic and semantic nuances into a single vector $\mathbf{h}_T \in \mathbb{R}^H$ causes severe information loss.
- Translation quality drops catastrophically as sentence length increases.

# The Attention Breakthrough
## Bahdanau Additive Attention (2014)

To resolve the bottleneck, Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio (2014) introduced **Attention**:
Instead of compressing the input into a single static vector, the decoder dynamically accesses **all encoder hidden states** $\mathbf{h}_1, \dots, \mathbf{h}_T$ at every decoding step $i$:

1. **Alignment Score (Additive)**:
   $$e_{ij} = \mathbf{v}_a^\mathrm{T} \tanh\left(\mathbf{W}_a \mathbf{s}_{i-1} + \mathbf{U}_a \mathbf{h}_j\right)$$
   where $\mathbf{s}_{i-1}$ is the previous decoder state and $\mathbf{h}_j$ is the $j$-th encoder state.

2. **Attention Weights (Softmax Distribution)**:
   $$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^T \exp(e_{ik})}$$

3. **Dynamic Context Vector**:
   $$\mathbf{c}_i = \sum_{j=1}^T \alpha_{ij} \mathbf{h}_j$$

<small>📖 *Source: [Bishop, C. M., & Bishop, H. (2024). Deep Learning: Foundations and Concepts. Springer (Section 12.2)](https://doi.org/10.1007/978-3-031-45468-4)*</small>

In [ ]:
# PyTorch LSTM Sequence Forecasting Implementation
class SequenceLSTM(nn.Module):
    def __init__(self, in_dim=1, hidden_dim=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden_dim, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])

# Generate synthetic sinusoidal time series
torch.manual_seed(42)
t_steps = torch.linspace(0, 10 * math.pi, 300)
series = torch.sin(t_steps) + torch.randn(300) * 0.1

# Prepare sequence windows (window_size = 15)
window_size = 15
X_seq, y_seq = [], []
for i in range(len(series) - window_size):
    X_seq.append(series[i:i+window_size])
    y_seq.append(series[i+window_size])

X_seq = torch.stack(X_seq).unsqueeze(-1)
y_seq = torch.stack(y_seq).unsqueeze(-1)

model_seq = SequenceLSTM(in_dim=1, hidden_dim=32)
optimizer = torch.optim.Adam(model_seq.parameters(), lr=0.01)
criterion = nn.MSELoss()

for epoch in range(100):
    optimizer.zero_grad()
    loss = criterion(model_seq(X_seq), y_seq)
    loss.backward()
    optimizer.step()

print(f"Final LSTM Sequence Training MSE Loss: {loss.item():.4f}")

In [ ]:
# Plotting helper for sequence prediction
def plot_sequence_forecast(lookback=15):
    with torch.no_grad():
        preds = model_seq(X_seq).squeeze().numpy()
        
    plt.figure(figsize=(8, 3.8))
    plt.plot(t_steps[lookback:].numpy(), y_seq.squeeze().numpy(), label="True Series", color="royalblue", alpha=0.7)
    plt.plot(t_steps[lookback:].numpy(), preds, label="LSTM Forecast", color="firebrick", lw=2)
    plt.title("LSTM Time Series Forecasting", fontsize=12)
    plt.xlabel("Time Step")
    plt.ylabel("Signal Value")
    plt.legend(loc='lower left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive display of sequence forecasting
interact(plot_sequence_forecast, lookback=fixed(15));

# Summary: Sequence Models & Attention Evolution

1. **Recurrence & Vanishing Gradients**:
   - Standard Elman RNNs propagate state via $\mathbf{h}_t = \tanh(\mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{W}_{xh}\mathbf{x}_t)$.
   - BPTT causes exponential gradient decay, rendering vanilla RNNs unable to bridge long temporal gaps.

2. **Gated Architectures (LSTM & GRU)**:
   - Additive memory carousels $\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$ ensure stable gradient transmission.

3. **The Attention Breakthrough**:
   - The fixed-vector bottleneck in Seq2Seq motivated **Bahdanau Additive Attention (2014)**.
   - Dynamic context vectors $\mathbf{c}_i = \sum \alpha_{ij} \mathbf{h}_j$ paved the direct evolutionary pathway to pure Self-Attention in Transformers.